<a href="https://colab.research.google.com/github/sakshammishra2112-byte/flyrank-ml/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

**Name:** Saksham Mishra  
**Lane:** Refresh / Content Opportunity Scoring

This notebook frames the lane before model training. The goal is to improve a real content-review decision: **which pages should a content reviewer inspect first when review capacity is limited?**

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring.**

The output should be a continuous **review-priority score** for each content page, so pages can be ordered from higher to lower review priority. This matches the decision in my Week 1 research question: a content team needs to decide **which pages to review first**, not simply assign every page to a yes/no class.

The person acting on the output is a **content reviewer/editor**. The reviewer uses the ranked queue to inspect high-priority pages and decide whether to refresh, expand, protect, prune, or continue monitoring them.

The intended output is therefore a **ranking/score**, while the eventual learning target should be an observed outcome rather than a hand-written priority rule.

## 2. Target or proxy

The ideal target is an **observed future content outcome** measured after the scoring window. For example, after a page is reviewed and an eligible intervention is made, the eventual outcome could measure improvement in search visibility or traffic over a defined future window.

The starter CSV does not contain a future post-refresh outcome, so it cannot honestly provide that final target. For this framing exercise, I use the observed `trend_direction` as a **proxy for current performance pressure** and show its binary shape as `is_declining_proxy`.

Important: `trend_direction` and `trend_pct` are derived from the same recent-vs-previous window and **must not be used as model features**. They are used here only to demonstrate what an observed proxy/label can look like.

The final warehouse version should replace this proxy with a future-window outcome that is observed after the feature window.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

proxy = (df["trend_direction"] == "down").astype("int8")
print("\nObserved proxy: current decline")
print(proxy.value_counts().rename(index={0: "not_down", 1: "down"}))
print("\nProxy rate:", round(proxy.mean() * 100, 1), "%")

Rows: 30000
Columns: 44

Observed proxy: current decline
trend_direction
down        16262
not_down    13738
Name: count, dtype: int64

Proxy rate: 54.2 %


## 3. Success metric

**Primary metric: Precision@10.**

The practical use case is a ranked review queue. Precision@10 asks: **among the first 10 pages shown to reviewers, how many are genuinely useful opportunities according to the observed evaluation outcome?**

A higher Precision@10 means fewer of the very first review slots are spent on pages that the evaluation labels as non-opportunities. This metric is appropriate because the content team has limited review capacity and the top of the queue matters most.

For the final model, Precision@10 must be calculated against a **future observed outcome**. It should not be calculated by treating the model's own scoring rule as the ground truth.

## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page.**

The starter dataset contains 30,000 content-item rows and 44 columns. The relevant signals include search activity, engagement, content age/freshness, and content metadata.

The identifiers `content_id` and `client_id` are used for grouping or joins only; they are not model features.

The data dictionary also warns that `trend_direction` and `trend_pct` are label-derived fields, so they are excluded from candidate features.

In [ ]:
# Show the actual unit of analysis and a small set of candidate observable signals.
candidate_columns = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "ai_sessions_90d",
]

display(df[candidate_columns].head(10))

print("Unit of analysis: one row = one content page.")
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,ai_sessions_90d
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,0.76,10.6,5.88,0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,0.05,20.3,0.00,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,0.09,36.5,0.00,0
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,0.49,6.2,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,0.13,44.0,0.00,0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,0.03,8.5,0.00,0
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,1,0.00,7.0,0.00,0
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,0.06,21.2,3.57,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,0.09,46.0,5.88,0
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,3,0.16,4.9,0.00,0


Unit of analysis: one row = one content page.
Unique content IDs: 30000
Unique clients: 32


### Target-column sketch

For the eventual warehouse model, I would add an outcome column that is measured **after** the feature window, for example:

`future_30d_refresh_outcome`

The exact definition should be fixed in the later data-contract/validation work. One possible shape is a binary observed outcome such as:

- `1` = the page meets the pre-declared future outcome threshold
- `0` = it does not

The starter CSV cannot populate this future column because it is a trailing-90-day snapshot. The current `trend_direction` proxy is therefore useful for framing and label-shape inspection, but it is not a substitute for a future causal or post-action outcome.

In [ ]:
# Sketch the eventual target schema without pretending the starter snapshot contains future labels.
target_sketch = df[[
    "content_id",
    "client_id",
    "trend_direction"
]].head(10).copy()

target_sketch["is_declining_proxy"] = (
    target_sketch["trend_direction"] == "down"
).astype("int8")

target_sketch["future_30d_refresh_outcome"] = pd.Series(
    [pd.NA] * len(target_sketch), dtype="Int64"
)

display(target_sketch)

,content_id,client_id,trend_direction,is_declining_proxy,future_30d_refresh_outcome
0,content_304f48230142,client_f369cb89fc,down,1,<NA>
1,content_a1fb4e703a9e,client_4e07408562,down,1,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,down,1,<NA>
3,content_331d6c4de07b,client_19581e27de,stable,0,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,down,1,<NA>
5,content_d4084a4bc775,client_f369cb89fc,down,1,<NA>
6,content_9a34b442b552,client_8722616204,down,1,<NA>
7,content_a63219c6e95a,client_19581e27de,stable,0,<NA>
8,content_5e6c160719bc,client_6208ef0f77,down,1,<NA>
9,content_c27558df2b0c,client_19581e27de,down,1,<NA>


## 5. Why ML beats a fixed rule here

A fixed rule can be a useful baseline, but the review-priority pattern can involve several signals at once.

For example, two pages may both have falling impressions, but one may have enough search demand and engagement to justify a review while the other has very little measurable activity. Likewise, pages can differ in content age, freshness, CTR, clicks, sessions, and search position.

A single rule such as `if trend_pct < -20%: refresh` cannot naturally represent all of these interactions without accumulating many hand-tuned thresholds.

ML earns its place if validation shows that combining these observable signals improves the top of the review queue over a simple baseline. If it does not, a rule or dashboard would be the more appropriate solution.

**Decision and action:** the score supports a content reviewer in choosing what to inspect first. It does not automatically publish, refresh, prune, or otherwise change content.

In [ ]:
# Basic framing checks: candidate feature columns do not include the label-derived trend fields.
candidate_feature_columns = [
    "content_type",
    "main_intent",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

leakage_columns = {"trend_direction", "trend_pct", "is_declining_label"}
print("Candidate feature count:", len(candidate_feature_columns))
print("Leakage-prone label-derived columns excluded:",
      sorted(leakage_columns.intersection(candidate_feature_columns)))

Candidate feature count: 23
Leakage-prone label-derived columns excluded: []


## Self-check

- [x] **Task type:** ranking / scoring
- [x] **Decision:** which content pages should a reviewer inspect first?
- [x] **Actor + action:** content reviewer/editor uses the ranked queue to inspect and choose an appropriate content action
- [x] **Target/proxy:** observed current decline is shown only as a starter-data proxy; the final target should be a future observed outcome
- [x] **Success metric:** Precision@10, defined before model training
- [x] **Unit of analysis:** one row = one pseudonymized content page
- [x] **Actual dataframe:** loaded from `data/raw/content_refresh_anonymized.csv`
- [x] **Leakage check:** `trend_direction` and `trend_pct` are not candidate features
- [x] **Why ML:** multiple interacting signals may be difficult to encode with a small set of fixed thresholds; validation must prove whether ML actually helps
- [x] **Careful claim:** the output is decision support, not a guarantee that refreshing a page will improve performance

**Status:** The starter snapshot is sufficient to frame the problem and demonstrate the unit of analysis. The future outcome label and final evaluation belong in the later warehouse/data-contract and validation work.